# 02 · MLP로 Breast Cancer 분류

Layer 1의 XGBoost와 **같은 데이터, 다른 모델**.
여기서 진짜 배울 것:
- `nn.Module`로 모델 작성
- `Dataset` + `DataLoader`
- 표준 학습 루프 (`optimizer.zero_grad()` → `loss.backward()` → `optimizer.step()`)
- train / eval 모드 전환
- 학습 곡선 해석

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score

torch.manual_seed(42)
np.random.seed(42)

device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'device = {device}')

## 1. 데이터 준비

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_tr = scaler.fit_transform(X_tr)
X_te = scaler.transform(X_te)

print(f'train {X_tr.shape}, test {X_te.shape}, n_features = {X_tr.shape[1]}')

## 2. Dataset & DataLoader

`Dataset`은 "하나의 샘플을 어떻게 꺼낼지" 정의. `DataLoader`는 **배치 구성 + shuffling + 병렬 로딩**을 담당.

In [ ]:
class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_ds = TabularDataset(X_tr, y_tr)
test_ds  = TabularDataset(X_te, y_te)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=64, shuffle=False)

# 한 배치 미리보기
xb, yb = next(iter(train_loader))
print(f'배치 X {xb.shape}, y {yb.shape}')

## 3. 모델 정의 — MLP

30개 feature → hidden 64 → hidden 32 → logit 1 (이진 분류).

In [ ]:
class MLP(nn.Module):
    def __init__(self, in_dim, hidden=(64, 32), dropout=0.2):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))   # logit
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)   # (B, 1) → (B,)

model = MLP(in_dim=X_tr.shape[1]).to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f'파라미터 수: {total_params:,}')

## 4. Loss & Optimizer

- `BCEWithLogitsLoss`: 이진 분류용. Sigmoid + BCE를 수치적으로 안정적으로 합친 것.
- `Adam`: 거의 기본값. learning rate 1e-3.

In [ ]:
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

## 5. 학습 루프

매 epoch 마다:
1. **train 모드** → batch마다 forward/backward/step
2. **eval 모드** → test셋에서 loss, AUROC 측정

In [ ]:
def evaluate(model, loader):
    model.eval()
    all_p, all_y, loss_sum, n = [], [], 0.0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss_sum += loss.item() * len(yb); n += len(yb)
            all_p.append(torch.sigmoid(logits).cpu().numpy())
            all_y.append(yb.cpu().numpy())
    proba = np.concatenate(all_p); y_true = np.concatenate(all_y)
    return loss_sum / n, roc_auc_score(y_true, proba), accuracy_score(y_true, proba > 0.5)

EPOCHS = 30
history = {'train_loss': [], 'test_loss': [], 'test_auc': []}

for epoch in range(1, EPOCHS + 1):
    model.train()
    running, n = 0.0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()
        running += loss.item() * len(yb); n += len(yb)
    train_loss = running / n
    test_loss, test_auc, test_acc = evaluate(model, test_loader)
    history['train_loss'].append(train_loss)
    history['test_loss'].append(test_loss)
    history['test_auc'].append(test_auc)
    if epoch % 5 == 0 or epoch == 1:
        print(f'epoch {epoch:3d} | train_loss {train_loss:.4f} | test_loss {test_loss:.4f} | test_auc {test_auc:.4f} | test_acc {test_acc:.4f}')

## 6. 학습 곡선

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history['train_loss'], label='train')
axes[0].plot(history['test_loss'], label='test')
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('loss'); axes[0].legend(); axes[0].set_title('Loss')

axes[1].plot(history['test_auc'])
axes[1].set_xlabel('epoch'); axes[1].set_ylabel('AUROC'); axes[1].set_title('Test AUROC')
plt.tight_layout()
plt.show()

## 7. 인사이트

**관찰**:
- 30 epoch만에 XGBoost에 근접하는 AUROC가 나오면 합격.
- train_loss < test_loss 간격이 크면 과적합 → `dropout`↑, `weight_decay`↑, 또는 layer 더 얕게.
- 작은 데이터(569개)에서는 **딥러닝이 XGBoost를 크게 못 이기는 경우가 많다** — 도메인 감각.

**AI agent에 물어볼 것**:
1. "BCEWithLogitsLoss와 BCELoss + sigmoid를 따로 쓰는 것의 수치적 차이가 뭐야?"
2. "Dropout을 eval 모드에서 꺼야 하는 이유를 코드 동작 관점에서 설명해줘"
3. "이 모델의 hidden을 (256, 128, 64)로 키워보고 과적합이 어떻게 달라지는지 실험해줘"